# 02 — VQE Ground State

> **spinq-vqe** | ARPA Quantum Logical Systems (QONDRA)

Ground-state energy of the 9-site Kagome Heisenberg AFM via VQE.

**Target:** $E_0 = -1.42190399$ (exact diagonalization, Notebook 01)

## Optimizer choice

Adam (gradient-based) stalls on this system: the $|0\rangle^{\otimes N}$ initial
state is a Z-basis eigenstate. All IsingXX/YY/ZZ and RY/CNOT gradients cancel
to zero there by SU(2) symmetry — the optimizer has no signal to follow.

**COBYLA** (gradient-free, SciPy) builds a local linear model from energy
evaluations. It does not need gradients, naturally explores the landscape,
and finds the ground state region reliably for this system size.

| | COBYLA (primary) | Adam (diagnostic) |
|-|-----------------|-------------------|
| Needs gradients | No | Yes |
| Escapes local minima | ✅ Better | ❌ Stalls |
| Barren plateau immune | ✅ | ❌ |
| Cost per step | 1 energy eval | 1 energy + 1 gradient |

### References
- Peruzzo et al. (2014) Nature Commun. 5, 4213 — original VQE
- Kandala et al. (2017) Nature 549, 242 — hardware-efficient ansatz
- McClean et al. (2018) Nat. Commun. 9, 4812 — barren plateaus
- Consiglio et al. (2022) PRX Research 4, 033257 — VQE on Kagome

In [ ]:
from __future__ import annotations
import os, csv, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qp

from spinq_vqe import kagome, ansatz, vqe

os.makedirs('../figures', exist_ok=True)
os.makedirs('../data', exist_ok=True)

PAL = {'cobyla': '#7EB8D4', 'adam': '#E8A598', 'ed': '#B0B0B0'}
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'figure.facecolor': 'white', 'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#EBEBEB',
    'grid.linewidth': 0.7, 'figure.dpi': 110,
})
print('PennyLane', qp.__version__)

---
## 1. Setup

In [ ]:
with open('../data/ed_reference_energies.csv') as f:
    E0_ref = float(next(r['E0_normalized'] for r in csv.DictReader(f) if r['n_sites'] == '9'))
print(f'ED reference  E₀(N=9) = {E0_ref:.8f}')

G9      = kagome.kagome_graph(n_cells=3)
H9      = kagome.heisenberg_kagome_hamiltonian(G9)
N_SITES = kagome.n_sites(G9)
edges   = list(G9.edges())
DEPTH   = 3
SEEDS   = [42, 7, 123, 99, 17]
N_PARAMS = ansatz.hea_n_params(N_SITES, DEPTH)

print(f'Sites={N_SITES}  Bonds={kagome.n_bonds(G9)}  Pauli terms={len(H9.coeffs)}')
print(f'HEA depth={DEPTH}  params={N_PARAMS}')

---
## 2. COBYLA — Primary Optimizer

5000 energy evaluations per seed × 5 seeds, keeping the best result.
Uses HEA depth=3 (27 parameters) with random initialisation.

In [ ]:
cobyla_runs = []
for seed in SEEDS:
    p0 = ansatz.init_params('hea', N_SITES, depth=DEPTH, seed=seed, scale=1.0)
    result = vqe.run_vqe_cobyla(
        hamiltonian=H9,
        ansatz_fn=ansatz.hea_ansatz,
        init_params=p0,
        n_sites=N_SITES,
        ansatz_name='hea',
        n_evals=5000,
        rhobeg=0.5,
        return_statevector=True,
        verbose=False,
        edges=edges,
        depth=DEPTH,
    )
    cobyla_runs.append(result)
    err = abs(result.energy - E0_ref) / abs(E0_ref) * 100
    print(f'  seed={seed:>3}  E={result.energy:+.6f}  err={err:.2f}%  evals={result.n_steps}')

best = min(cobyla_runs, key=lambda r: r.energy)
err_best = abs(best.energy - E0_ref) / abs(E0_ref) * 100
print(f'\nBest COBYLA/HEA: {best.energy:+.8f}  ({err_best:.2f}% error from ED)')

---
## 3. Adam — Diagnostic Comparison

Run Adam on the same ansatz / same seeds for comparison and barren plateau
analysis. Expected to stall due to zero gradients near Z-basis eigenstates.

In [ ]:
adam_runs = []
for seed in SEEDS[:2]:  # only 2 seeds — diagnostic, not primary
    p0 = ansatz.init_params('hea', N_SITES, depth=DEPTH, seed=seed, scale=1.0)
    result = vqe.run_vqe(
        hamiltonian=H9,
        ansatz_fn=ansatz.hea_ansatz,
        init_params=p0,
        n_sites=N_SITES,
        ansatz_name='hea',
        n_steps=1000,
        return_statevector=False,
        verbose=False,
        edges=edges,
        depth=DEPTH,
    )
    adam_runs.append(result)
    err = abs(result.energy - E0_ref) / abs(E0_ref) * 100
    print(f'  seed={seed:>3}  E={result.energy:+.6f}  err={err:.2f}%  steps={result.n_steps}')

adam_best = min(adam_runs, key=lambda r: r.energy)

---
## 4. Results

In [ ]:
cols = ['Method', 'Ansatz', 'Params', 'Best E', 'Err %', 'Evals']
rows = [
    {'Method': 'COBYLA',    'Ansatz': 'HEA d=3', 'Params': N_PARAMS,
     'Best E': round(best.energy, 8), 'Err %': round(err_best, 3), 'Evals': best.n_steps},
    {'Method': 'Adam',      'Ansatz': 'HEA d=3', 'Params': N_PARAMS,
     'Best E': round(adam_best.energy, 8),
     'Err %': round(abs(adam_best.energy - E0_ref)/abs(E0_ref)*100, 3),
     'Evals': adam_best.n_steps},
    {'Method': 'ED (exact)','Ansatz': '--',       'Params': '--',
     'Best E': round(E0_ref, 8), 'Err %': 0.0,  'Evals': '--'},
]

print('=== VQE Results — 9-site Kagome Heisenberg AFM ===')
print('  '.join(f'{c:>14}' for c in cols))
print('─' * 90)
for row in rows:
    print('  '.join(f'{str(row[c]):>14}' for c in cols))

with open('../data/vqe_results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(rows)
print('\nSaved → data/vqe_results.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# COBYLA convergence (energy vs evaluation)
ax = axes[0]
for r in cobyla_runs:
    ax.plot(r.energy_history, color=PAL['cobyla'], alpha=0.2, lw=0.8)
ax.plot(best.energy_history, color=PAL['cobyla'], lw=2,
        label=f'COBYLA best {best.energy:+.5f}  ({err_best:.1f}%)')
ax.axhline(E0_ref, color=PAL['ed'], ls='--', lw=1.4, label=f'ED exact {E0_ref:+.5f}')
ax.set_xlabel('Function evaluation')
ax.set_ylabel('Energy (normalized)')
ax.set_title('COBYLA Convergence\n9-site Kagome Heisenberg AFM',
             fontweight='semibold', color='#333333')
ax.legend(fontsize=9, framealpha=0.9)

# Adam gradient variance (barren plateau diagnostic)
ax2 = axes[1]
for r in adam_runs:
    ax2.semilogy(r.gradient_variance_history, color=PAL['adam'], alpha=0.6, lw=1.5)
ax2.set_xlabel('Adam step')
ax2.set_ylabel('Gradient variance (log scale)')
ax2.set_title('Adam Gradient Variance\n(Barren Plateau Diagnostic)',
              fontweight='semibold', color='#333333')
ax2.text(0.05, 0.1, 'Vanishing gradient\u2192 Adam stalls', transform=ax2.transAxes,
         fontsize=9, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('../figures/vqe_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/vqe_convergence.png')

# Bar chart
fig2, ax3 = plt.subplots(figsize=(6, 4))
labels = ['COBYLA/HEA', 'Adam/HEA', 'ED (exact)']
values = [best.energy, adam_best.energy, E0_ref]
colors = [PAL['cobyla'], PAL['adam'], PAL['ed']]
bars = ax3.bar(labels, values, color=colors, edgecolor='#CCCCCC', lw=0.8, width=0.5)
for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, val - 0.03,
             f'{val:.4f}', ha='center', va='top', fontsize=9, color='#444')
ax3.axhline(E0_ref, color=PAL['ed'], ls='--', lw=1.2, alpha=0.6)
ax3.set_ylabel('Ground state energy (normalized)')
ax3.set_title('VQE vs Exact Diagonalization\n9-site Kagome AFM',
              fontweight='semibold', color='#333333')
ax3.set_ylim(min(values) - 0.1, 0.2)
plt.tight_layout()
plt.savefig('../figures/vqe_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/vqe_bar.png')

---
## 5. Save statevector → Notebook 03

In [ ]:
np.save('../data/statevector_hea_best.npy', best.statevector)
print(f'Statevector shape: {best.statevector.shape}')
print(f'Norm: {np.linalg.norm(best.statevector):.8f}  (should be 1.0)')
print(f'Best energy: {best.energy:.8f}')
print(f'Error from ED: {err_best:.2f}%')
print('Saved → data/statevector_hea_best.npy')

---
*spinq-vqe / ARPA QONDRA — Notebook 02*